# Morocco SCOE (Buildings) — Fuel-mix calibration

**Purpose.** Audit and correct the SCOE heat-energy fuel fractions in `sisepuede_raw_input_morocco_fuels.csv` so they match Morocco's real buildings energy balance (residential + commercial/services + other stationary emissions).

**Why.** The current `sisepuede_raw_input_morocco_fuels.csv` carries placeholder/stylized values (e.g. residential heat: electricity 40%, LPG 30%, biomass 20%, NG 5%, kerosene 3%) that do not reflect Morocco's actual structure. Morocco's residential sector is dominated by **butane (LPG)** for cooking and water heating, with biomass (wood/charcoal) still important for rural heating, and almost no kerosene or natural gas in buildings.

**Sources used to justify the calibrated values.**

| Source | Year | Key figure |
|---|---|---|
| IEA — World Energy Balances (Morocco residential TFC) | 2015 | Oil products 98,281 TJ · Electricity 36,543 TJ · Biofuels & waste 24,400 TJ |
| IEA — World Energy Balances (Morocco commercial & public services TFC) | 2015 | Oil products 5,589 TJ · Electricity 18,331 TJ · Biofuels & waste 27,291 TJ |
| IEA — Morocco country page (Efficiency & Demand) | 2020 | Residential = 27% of TFC; LPG/butane > 2/3 of residential consumption (IEA average is ~13%) |
| Enerdata — Morocco Energy Market | 2020 | Butane = dominant residential fuel (share grew from 57% in 2010 to 66% in 2020); electricity 24%; biofuels/waste 11% (down from 23% in 2010) |
| AFREC — Africa Energy Database | 2018-2020 | TFC fuel shares: oil 40%, biomass (wood/charcoal/animal waste) 36%, NG 4%, electricity 20%; traditional biomass mainly rural |
| Our World in Data / energypedia | 2020 | Buildings = 33% of TFC (residential 26% + commercial 7%); rural off-grid relies on firewood for cooking, water and space heating |

**SISEPUEDE variable structure.** SCOE splits residential and commercial energy into two demand streams:
- `consumpinit_scoe_*_heat_energy` (cooking, water heating, space heating) — distributed by `frac_scoe_heat_energy_<cat>_<fuel>`
- `consumpinit_scoe_*_elec_appliances` (lighting, refrigeration, AC, electronics) — always electricity

So the residential **heat** mix is dominated by LPG + biomass (the IEA electricity TFC is mostly appliances and is handled by the separate appliances variable). We must convert IEA *consumption* shares into SISEPUEDE *demand* shares using the SCOE efficiency factors (see Eq. 2 in `mathdoc_energy.html`).

**This notebook.**
1. Loads `sisepuede_raw_input_morocco_fuels.csv` and audits the current SCOE heat fractions.
2. Computes Morocco-correct **demand** fractions from IEA 2015 consumption + SCOE η factors.
3. Writes a backup and saves the corrected file in place.

In [8]:
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path('/Users/fabianfuentes/git/ssp_morocco')
INPUT_DIR = REPO / 'ssp_modeling' / 'input_data'
REF_DIR = INPUT_DIR / 'reference'

INPUT_FILE = INPUT_DIR / 'sisepuede_raw_input_morocco_fuels.csv'
REF_RES = REF_DIR / 'iea_morocco_residential_tfc_by_source.csv'
REF_COM = REF_DIR / 'iea_morocco_commercial_tfc_by_source.csv'

SCOE_FUELS = [
    'coal', 'diesel', 'electricity', 'gasoline',
    'hydrocarbon_gas_liquids', 'hydrogen', 'kerosene',
    'natural_gas', 'solid_biomass',
]
SCOE_CATS = ['residential', 'commercial_municipal', 'other_se']

pd.options.display.float_format = '{:.4f}'.format
df = pd.read_csv(INPUT_FILE)
print(f'Loaded {INPUT_FILE.name}: {df.shape}  tp=[{df.time_period.min()}, {df.time_period.max()}]')

Loaded sisepuede_raw_input_morocco_fuels.csv: (56, 2442)  tp=[0, 55]


## 1 · Audit — current SCOE heat fractions

Show the fractions at `tp=0` for each SCOE category and verify they sum to 1.

In [9]:
def fraction_table(df_in, tp=0):
    rows = []
    row = df_in[df_in.time_period == tp].iloc[0]
    for cat in SCOE_CATS:
        for fuel in SCOE_FUELS:
            col = f'frac_scoe_heat_energy_{cat}_{fuel}'
            rows.append({'cat': cat, 'fuel': fuel, 'frac': float(row[col])})
    return (
        pd.DataFrame(rows)
        .pivot(index='fuel', columns='cat', values='frac')
        .reindex(SCOE_FUELS)
    )

current = fraction_table(df, tp=0)
print('Current SCOE heat fractions at tp=0 (file as loaded):')
display(current)
print('Column sums:', current.sum().to_dict())

Current SCOE heat fractions at tp=0 (file as loaded):


cat,commercial_municipal,other_se,residential
fuel,,,
coal,0.0000,0.0000,0.0000
diesel,0.0000,0.0000,0.0000
electricity,0.0000,0.0000,0.0000
gasoline,0.0000,0.0000,0.0000
hydrocarbon_gas_liquids,0.2157,0.0000,0.8440
hydrogen,0.0000,0.0000,0.0000
kerosene,0.0000,0.0000,0.0000
natural_gas,0.0000,0.0000,0.0000
solid_biomass,0.7843,0.0000,0.1560


Column sums: {'commercial_municipal': 0.9999999999999999, 'other_se': 0.0, 'residential': 0.9999999999999999}


## 2 · Compute Morocco-correct demand fractions from IEA balances

**Method.** IEA reports *final consumption* by fuel. SISEPUEDE's `frac_scoe_heat_energy_*` are *useful-energy demand* fractions. The conversion (see `mathdoc_energy.html` Eq. 2) is:
$$
\alpha^D_f = \frac{\alpha^C_f \cdot \eta_f}{\sum_{f'} \alpha^C_{f'} \cdot \eta_{f'}}
$$
where $\alpha^C_f$ is the consumption share of fuel $f$ and $\eta_f$ is the SCOE efficiency factor (from the template, also stored in the input file).

**Residential heat (IEA 2015):** electricity TFC is excluded from the *heat* split because it is treated as appliance demand by SISEPUEDE (handled by `consumpinit_*_elec_appliances`). Heat = oil products (assumed = LPG/butane) + biofuels & waste (= solid biomass).

**Commercial/services heat (IEA 2015):** same logic — heat = oil + biomass; electricity goes to appliances.

**Other stationary emissions:** keep null (SISEPUEDE convention; demand specified exogenously when needed).

In [10]:
# --- IEA 2015 final consumption (TJ) ---
iea_res = pd.read_csv(REF_RES)
iea_com = pd.read_csv(REF_COM)
print('Residential TFC 2015 (IEA):'); display(iea_res)
print('Commercial TFC 2015 (IEA):');  display(iea_com)

# Pull the heat-relevant fuels (LPG/butane via Oil products; biomass via Biofuels & waste)
def tfc_pair(df_ref):
    by_src = dict(zip(df_ref.iloc[:, 0], df_ref['Value']))
    oil = by_src['Oil products (predominantly LPG/butane)']
    bio = by_src['Biofuels and waste (wood, charcoal)']
    return oil, bio

res_oil, res_bio = tfc_pair(iea_res)
com_oil, com_bio = tfc_pair(iea_com)
print(f'\nResidential heat (TJ): LPG={res_oil:,}  biomass={res_bio:,}  total={res_oil+res_bio:,}')
print(f'Commercial  heat (TJ): LPG={com_oil:,}  biomass={com_bio:,}  total={com_oil+com_bio:,}')

Residential TFC 2015 (IEA):


,residential total final consumption by source in Morocco,Value,Year,Units
0,Oil products (predominantly LPG/butane),98281,2015,TJ
1,Electricity,36543,2015,TJ
2,"Biofuels and waste (wood, charcoal)",24400,2015,TJ
3,Natural gas,0,2015,TJ
4,Coal and coal products,0,2015,TJ
5,Solar thermal,0,2015,TJ


Commercial TFC 2015 (IEA):


,commercial and public services total final consumption by source in Morocco,Value,Year,Units
0,Oil products (predominantly LPG/butane),5589,2015,TJ
1,Electricity,18331,2015,TJ
2,"Biofuels and waste (wood, charcoal)",27291,2015,TJ
3,Natural gas,0,2015,TJ
4,Coal and coal products,0,2015,TJ
5,Solar thermal,0,2015,TJ



Residential heat (TJ): LPG=98,281  biomass=24,400  total=122,681
Commercial  heat (TJ): LPG=5,589  biomass=27,291  total=32,880


In [11]:
# --- SCOE efficiency factors (read from the file itself; fall back to template defaults if missing) ---
def get_eff(df_in, cat, fuel, default):
    col = f'efficfactor_scoe_heat_energy_{cat}_{fuel}'
    if col in df_in.columns:
        v = float(df_in[col].iloc[0])
        return v if v > 0 else default
    return default

EFF_DEFAULTS = {'hydrocarbon_gas_liquids': 0.94, 'solid_biomass': 0.70}

def demand_fractions(df_in, cat, cons_lpg_tj, cons_bio_tj):
    eff_lpg = get_eff(df_in, cat, 'hydrocarbon_gas_liquids', EFF_DEFAULTS['hydrocarbon_gas_liquids'])
    eff_bio = get_eff(df_in, cat, 'solid_biomass', EFF_DEFAULTS['solid_biomass'])
    total_cons = cons_lpg_tj + cons_bio_tj
    if total_cons <= 0:
        return {f: 0.0 for f in SCOE_FUELS}
    cons_lpg = cons_lpg_tj / total_cons
    cons_bio = cons_bio_tj / total_cons
    denom = cons_lpg * eff_lpg + cons_bio * eff_bio
    dem_lpg = cons_lpg * eff_lpg / denom
    dem_bio = cons_bio * eff_bio / denom
    out = {f: 0.0 for f in SCOE_FUELS}
    out['hydrocarbon_gas_liquids'] = dem_lpg
    out['solid_biomass'] = dem_bio
    print(f'  {cat}: cons LPG={cons_lpg:.4f} bio={cons_bio:.4f} | eff LPG={eff_lpg:.2f} bio={eff_bio:.2f} | demand LPG={dem_lpg:.4f} bio={dem_bio:.4f}')
    return out

print('Morocco-calibrated SCOE heat DEMAND fractions:\n')
target = {
    'residential':          demand_fractions(df, 'residential',          res_oil, res_bio),
    'commercial_municipal': demand_fractions(df, 'commercial_municipal', com_oil, com_bio),
    'other_se':             {f: 0.0 for f in SCOE_FUELS},  # exogenous, leave zero
}
target_tbl = pd.DataFrame(target).reindex(SCOE_FUELS)
print('\nNew fractions:'); display(target_tbl)
print('Sums:', target_tbl.sum().to_dict())

Morocco-calibrated SCOE heat DEMAND fractions:

  residential: cons LPG=0.8011 bio=0.1989 | eff LPG=0.94 bio=0.70 | demand LPG=0.8440 bio=0.1560
  commercial_municipal: cons LPG=0.1700 bio=0.8300 | eff LPG=0.94 bio=0.70 | demand LPG=0.2157 bio=0.7843

New fractions:


,residential,commercial_municipal,other_se
coal,0.0000,0.0000,0.0000
diesel,0.0000,0.0000,0.0000
electricity,0.0000,0.0000,0.0000
gasoline,0.0000,0.0000,0.0000
hydrocarbon_gas_liquids,0.8440,0.2157,0.0000
hydrogen,0.0000,0.0000,0.0000
kerosene,0.0000,0.0000,0.0000
natural_gas,0.0000,0.0000,0.0000
solid_biomass,0.1560,0.7843,0.0000


Sums: {'residential': 1.0, 'commercial_municipal': 0.9999999999999999, 'other_se': 0.0}


### Note on the `other_se` category

SISEPUEDE convention (per `attribute_cat_scoe.csv`): *"Other stationary emissions — demands must be exogenously specified"*. We leave its `frac_*` columns at zero; if/when an exogenous demand is added later (e.g. street lighting if disaggregated separately), the matching fraction(s) can be re-set explicitly.

## 3 · Sanity checks before writing

- Each row's per-category fractions sum to 1 (or to 0 for `other_se`).
- Only LPG (`hydrocarbon_gas_liquids`) and biomass (`solid_biomass`) are non-zero in residential and commercial — consistent with IEA Morocco data.
- The number of rows touched matches `len(df)` (the calibration is constant across the trajectory — `frac` evolution is handled by transformations, not by the baseline input).

In [12]:
for cat, fracs in target.items():
    s = sum(fracs.values())
    expected = 0.0 if cat == 'other_se' else 1.0
    assert abs(s - expected) < 1e-9, f'{cat} sums to {s}, expected {expected}'
print('OK — all category fractions sum correctly.')

OK — all category fractions sum correctly.


## 4 · Apply correction and save

We overwrite `frac_scoe_heat_energy_<cat>_<fuel>` for every row of the input. A timestamped backup of the original is written next to the file.

In [13]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
backup = INPUT_FILE.with_suffix(f'.csv.bak_preSCOEcalib_{ts}')
shutil.copyfile(INPUT_FILE, backup)
print(f'Backup written: {backup.name}')

df_out = df.copy()
for cat, fracs in target.items():
    for fuel, val in fracs.items():
        col = f'frac_scoe_heat_energy_{cat}_{fuel}'
        if col not in df_out.columns:
            print(f'  WARN: column missing, skipping: {col}')
            continue
        df_out[col] = val

df_out.to_csv(INPUT_FILE, index=False)
print(f'Wrote corrected file: {INPUT_FILE.name}  rows={len(df_out)}')

Backup written: sisepuede_raw_input_morocco_fuels.csv.bak_preSCOEcalib_20260525_112456
Wrote corrected file: sisepuede_raw_input_morocco_fuels.csv  rows=56


## 5 · Verify

In [14]:
df_check = pd.read_csv(INPUT_FILE)
after = fraction_table(df_check, tp=0)
print('SCOE heat fractions after calibration (tp=0):')
display(after)
print('Sums by category:', after.sum().to_dict())

# Confirm constant across the trajectory
first_row = df_check.iloc[0][[f'frac_scoe_heat_energy_{c}_{f}' for c in SCOE_CATS for f in SCOE_FUELS]]
last_row  = df_check.iloc[-1][[f'frac_scoe_heat_energy_{c}_{f}' for c in SCOE_CATS for f in SCOE_FUELS]]
if (first_row.values == last_row.values).all():
    print('OK — fractions constant across the trajectory (transformations will modify them at run time).')
else:
    print('WARNING — fractions vary by time period after writing; expected constant baseline.')

SCOE heat fractions after calibration (tp=0):


cat,commercial_municipal,other_se,residential
fuel,,,
coal,0.0000,0.0000,0.0000
diesel,0.0000,0.0000,0.0000
electricity,0.0000,0.0000,0.0000
gasoline,0.0000,0.0000,0.0000
hydrocarbon_gas_liquids,0.2157,0.0000,0.8440
hydrogen,0.0000,0.0000,0.0000
kerosene,0.0000,0.0000,0.0000
natural_gas,0.0000,0.0000,0.0000
solid_biomass,0.7843,0.0000,0.1560


Sums by category: {'commercial_municipal': 0.9999999999999999, 'other_se': 0.0, 'residential': 0.9999999999999999}
OK — fractions constant across the trajectory (transformations will modify them at run time).


## 6 · Notes & next steps

1. **Why no electricity in the residential heat fraction?** IEA 2015 reports residential electricity TFC = 36,543 TJ, but in SISEPUEDE that demand belongs to `consumpinit_scoe_gj_per_hh_residential_elec_appliances` (lighting, fridges, AC, etc.) — not to the heat split. Mixing electricity into the heat fraction would double-count appliance demand. If the *real* heat mix in 2050 includes electric water heaters / heat pumps, that share enters via the `TFR:SCOE:SHIFT_FUEL_HEAT` transformer, not via the baseline fractions.
2. **Updating to a more recent year.** The 2020 picture from IEA/Enerdata is qualitatively the same (butane > 2/3 of residential, biomass declining from ~23 % in 2010 to ~11 % in 2020). If/when we obtain IEA Balances 2022 in TJ for residential and commercial sectors, drop the new numbers into the two reference CSVs and re-run the notebook — the rest of the pipeline is unchanged.
3. **Commercial vs municipal disaggregation.** SISEPUEDE collapses commercial + municipal/public services into one category. Morocco's street-lighting electrification target lives inside this same category; we cannot separate it without a schema change. The TX:SCOE:INC_EFFICIENCY_APPLIANCE transformer will pick it up on the appliances side.
4. **Solar thermal (DHW panels).** Still no native `fuel_solar_thermal` in SCOE. The MW-equivalents from the SNBC targets must be modelled either as an additional heat-demand reduction (camera A) or via a future schema extension (camera B). This notebook does **not** touch that — solar thermal stays at 0 in the heat fractions.

## 7 · Workflow handoff — confirm the run pipeline picks this file up

The main workflow notebook ([`notebooks/workflow/morocco_manager_wb.ipynb`](../workflow/morocco_manager_wb.ipynb)) reads `ssp_input_file_name` from `notebooks/config_files/config.yaml`. We assert it equals the file we just wrote — if not, the run will use a different baseline and the calibration we just applied will be ignored.


In [ ]:
import yaml

CONFIG_FILE = REPO / 'ssp_modeling' / 'notebooks' / 'config_files' / 'config.yaml'
with open(CONFIG_FILE) as f:
    cfg = yaml.safe_load(f)

expected = INPUT_FILE.name
got = cfg.get('ssp_input_file_name')
print(f'config.yaml -> ssp_input_file_name: {got}')
print(f'this notebook wrote           : {expected}')
assert got == expected, (
    f'Mismatch: workflow will read {got}, not the calibrated {expected}. '
    f'Edit {CONFIG_FILE} so ssp_input_file_name == "{expected}".'
)
print('OK — the workflow run will use the calibrated baseline.')

## 8 · Save calibration snapshot for audit trail

Writes a timestamped CSV under `input_data/reference/` documenting (a) the before/after `frac_scoe_heat_energy_*` values, (b) the IEA source numbers used, and (c) the file modified. Useful for future calibrations to see *what changed and why* without diffing the full input CSV.


In [ ]:
snapshot_rows = []
for cat in SCOE_CATS:
    for fuel in SCOE_FUELS:
        col = f'frac_scoe_heat_energy_{cat}_{fuel}'
        snapshot_rows.append({
            'category': cat,
            'fuel': fuel,
            'frac_before': float(current.loc[fuel, cat]),
            'frac_after':  float(target[cat][fuel]),
        })
snapshot = pd.DataFrame(snapshot_rows)
snapshot['delta'] = snapshot['frac_after'] - snapshot['frac_before']

# Provenance header
meta = pd.DataFrame([
    {'category':'_meta', 'fuel':'timestamp',          'frac_before':None, 'frac_after':None, 'delta':ts},
    {'category':'_meta', 'fuel':'notebook',           'frac_before':None, 'frac_after':None, 'delta':'buildings/scoe_mix.ipynb'},
    {'category':'_meta', 'fuel':'input_file_modified','frac_before':None, 'frac_after':None, 'delta':INPUT_FILE.name},
    {'category':'_meta', 'fuel':'backup',             'frac_before':None, 'frac_after':None, 'delta':backup.name},
    {'category':'_meta', 'fuel':'res_oil_TJ_2015_IEA','frac_before':None, 'frac_after':None, 'delta':str(res_oil)},
    {'category':'_meta', 'fuel':'res_bio_TJ_2015_IEA','frac_before':None, 'frac_after':None, 'delta':str(res_bio)},
    {'category':'_meta', 'fuel':'com_oil_TJ_2015_IEA','frac_before':None, 'frac_after':None, 'delta':str(com_oil)},
    {'category':'_meta', 'fuel':'com_bio_TJ_2015_IEA','frac_before':None, 'frac_after':None, 'delta':str(com_bio)},
])
snapshot_full = pd.concat([meta, snapshot], ignore_index=True)

snap_path = REF_DIR / f'scoe_mix_calibration_{ts}.csv'
snapshot_full.to_csv(snap_path, index=False)
print(f'Snapshot saved: {snap_path.relative_to(REPO)}')
display(snapshot[snapshot['delta'].abs() > 1e-6])

## 9 · Next step — run the model

The baseline is now Morocco-calibrated and saved. To execute a run with the corrected SCOE mix:

1. Open [`notebooks/workflow/morocco_manager_wb.ipynb`](../workflow/morocco_manager_wb.ipynb).
2. Run all cells (it will load `sisepuede_raw_input_morocco_fuels.csv` per `config.yaml`).
3. Output lands in `ssp_modeling/ssp_run_output/sisepuede_results_<timestamp>/`.
4. Inspect `frac_scoe_heat_energy_residential_*` in the WIDE_INPUTS_OUTPUTS CSV — at `time_period=0` you should see LPG ≈ 0.844 and biomass ≈ 0.156 (the new baseline). Anything else means the workflow is reading from a different file.

**Rollback.** If you need the previous baseline back:
```bash
cp ssp_modeling/input_data/sisepuede_raw_input_morocco_fuels.csv.bak_preSCOEcalib_<timestamp> \\
   ssp_modeling/input_data/sisepuede_raw_input_morocco_fuels.csv
```